# Prophet Forecasting

Continues from [`2_forecasting_models.ipynb`](2_forecasting_models.ipynb): forecasts the same NVIDIA Close / semiconductor billings series, this time using Facebook/Meta's [Prophet](https://facebook.github.io/prophet/) library instead of classical statsmodels methods (exponential smoothing, ARIMA/SARIMA/SARIMAX).

**Dependency check:** does this notebook need anything `2_forecasting_models.ipynb` derived that isn't already in `data/final.pkl` from `1_data_preparation.ipynb`?

`2_` adds a few things on top of `final.pkl` -- `Returns` (`Close.pct_change()`), `Value_diff` (`Value_num.diff()`), a `Date_x` DatetimeIndex, and a `dropna()` -- but all of that exists specifically to make the series **stationary** for ARIMA/SARIMAX, which assumes stationarity. Prophet fits trend + seasonality directly on the **raw, non-stationary** series, so none of `2_`'s transformations are actual inputs it needs.

**Conclusion:** no new dependency needs saving from `2_` -- `data/final.pkl` (as written by `1_`) already has everything this notebook needs (`Date_x`, `Close`, `Value_num`).

## Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet # Facebook/Meta's forecasting library -- fits trend + seasonality + holiday effects directly, no manual differencing/ACF-PACF tuning required
from prophet.diagnostics import cross_validation, performance_metrics # rolling-origin cross-validation and MAE/RMSE/MAPE scoring for a fitted Prophet model
from prophet.plot import plot_cross_validation_metric, plot_plotly, plot_components_plotly # plotting helpers for CV metrics and interactive forecast/component plots

final = pd.read_pickle("data/final.pkl") # prepared dataframe from 1_data_preparation.ipynb -- same source 2_forecasting_models.ipynb loads (see dependency check above: no extra data needed from 2_)
final.head()

Importing plotly failed. Interactive plots will not work.


,Date_x,Close,High,Low,Open,Volume,Close_lag1,Close_lag2,rolling_mean_3_x,rolling_sd_3_x,...,expanding_mean_y,Value_growth,Year_y,Month_y,log_Close,log_Value,boxcox_Close,diff_Close,diff_Value,seasonal_diff_Close
12,2023-04-03,27.907740,27.942668,27.280028,27.452674,398716000,27.720123,27.326929,27.651597,0.296407,...,19.645842,-0.114619,2023,4,3.328904,2.175433,6.241134,5.260063,-1.624,1.275326
13,2023-05-01,28.850805,28.998503,27.723120,27.782996,570329000,27.692184,27.170252,27.904414,0.860143,...,19.938811,-0.191382,2023,5,3.362138,2.297170,6.346521,0.943066,1.140,9.376001
14,2023-06-01,39.688572,39.967997,38.261500,38.410193,635873000,37.756531,40.028870,39.157991,1.225569,...,20.216389,0.144186,2023,6,3.681063,2.509599,7.421725,10.837767,2.354,21.423155
15,2023-07-03,42.330528,42.814586,42.119940,42.434326,198209000,42.219753,40.742630,41.764304,0.886527,...,20.442571,-0.063589,2023,7,3.745509,2.374906,7.653731,2.641956,-1.550,27.847735
16,2023-08-01,46.416576,46.808814,45.937510,46.369667,237858000,46.638149,46.659100,46.571275,0.134382,...,20.727647,-0.126332,2023,8,3.837657,2.440606,7.994558,4.086048,0.730,28.026630


### Preparing data for Prophet

Prophet requires an input dataframe with exactly two columns: **`ds`** (datestamp) and **`y`** (the value to forecast) -- unlike the DatetimeIndex-based series used for statsmodels in `2_`.

In [2]:
prophet_df = final[['Date_x', 'Close']].rename(columns={'Date_x': 'ds', 'Close': 'y'}) # Prophet's required column names
prophet_df.head()

,ds,y
12,2023-04-03,27.907740
13,2023-05-01,28.850805
14,2023-06-01,39.688572
15,2023-07-03,42.330528
16,2023-08-01,46.416576
